# Hand gestures

A tiny classifier on top of the hand landmarks the vision pipeline already produces
(`tui/src/pi/track.ts`). It never sees pixels, only 21 points per frame, so it is ~10K weights
(~50 KB) and runs in microseconds anywhere ONNX runs.

One ONNX file, two heads:

| | name | shape | |
|---|---|---|---|
| in | `landmarks` | f32 [1, 21, 2] | camera pixels, MediaPipe order |
| in | `trail` | f32 [1, 16, 3] | last 16 frames of (palm centre x, y, hand span) in pixels, oldest first, left-padded with the oldest |
| out | `gesture` | f32 [1, 12] | softmax over `GESTURES` |
| out | `motion` | f32 [1, 6] | softmax over `MOTIONS` |

Normalisation lives inside the graph, so the API and the Pi feed raw pixels and can never drift
from training. Labels go in `hand_gesture.json` next to the model.

Run from `models/`: `uv sync`, then pick the `.venv` kernel.

In [1]:
import json
import math
import os
import shutil
import time
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import onnxruntime as ort
import torch
from torch import nn

HERE = Path.cwd()
assert (HERE / "pyproject.toml").exists(), "run this notebook from models/"
DATA = Path(os.environ.get("GESTURE_DATA", HERE / "data"))
OUT = HERE / "export"

GESTURES = ["none", "pinch", "palm", "fist", "point", "peace", "thumbs_up", "thumbs_down", "ok", "three", "call", "rock"]
MOTIONS = ["none", "swipe_left", "swipe_right", "swipe_up", "swipe_down", "wave"]
WINDOW = 16  # trail frames: ~0.5 s at 30 fps, ~1 s over the network

USE_HAGRID = True
HAGRID_PER_CLASS = 3000
HAGRID_URL = "https://rndml-team-cv.obs.ru-moscow-1.hc.sbercloud.ru/datasets/hagrid_v2/annotations_with_landmarks/annotations.zip"
# HaGRID class → ours. Anything not listed is dropped. No pinch in HaGRID: that one is all yours.
HAGRID_MAP = {
    "no_gesture": "none",
    "palm": "palm", "stop": "palm",
    "fist": "fist",
    "one": "point", "point": "point",
    "peace": "peace", "peace_inverted": "peace",
    "like": "thumbs_up", "dislike": "thumbs_down",
    "ok": "ok",
    "three": "three", "three2": "three",
    "call": "call",
    "rock": "rock",
}

SEED = 0
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
NONE_MOTION = MOTIONS.index("none")

## 1. Record

The vision worker already prints one JSON line per camera frame, so a recording is a shell
redirect. Once, from the repo root:

```
cd tui && bun add --no-save onnxruntime-node && bun src/index.tsx pi models && cd ..
```

Then one file per take (Ctrl-C to stop; on the Pi leave out `--source=ffmpeg`):

```
mkdir -p models/data/static/fist
bun tui/src/pi/vision-worker.ts --source=ffmpeg > models/data/static/fist/mac-01.ndjson
```

- **`data/static/<gesture>/`**: 2–3 takes of ~45 s. Hold the pose and drift it *slowly* around
  the frame; vary distance, tilt and which hand. `none`: relaxed hands, half-made poses, holding tools.
- **`data/motion/<motion>/`**: ~40 repetitions per take. Make the motion, drop the hand out of
  view, repeat. Each stretch with a hand in view is one clip. `none`: hovering, slow
  repositioning, drawing with a pinch.
- The **last file by name** in each folder is held out for validation. Make it a take from the
  real camera (the Pi), e.g. `pi-99.ndjson`.
- Swipe directions are **as the camera sees them**. A laptop webcam is not mirrored, so moving
  your hand to your right is a `swipe_left` there. The check below warns when a folder disagrees.

In [2]:
def read_ndjson(path):
    # (t, landmarks [21, 2] or None) per frame. Skips the worker's ready/error lines and a
    # last line cut off by Ctrl-C.
    frames = []
    for line in path.read_text().splitlines():
        try:
            row = json.loads(line)
        except json.JSONDecodeError:
            continue
        if "hands" not in row:
            continue
        hand = row["hands"][0] if row["hands"] else None
        points = np.array([[p["x"], p["y"]] for p in hand["landmarks"]], np.float32) if hand else None
        frames.append((row["t"], points))
    return frames


def takes(kind, labels):
    # {label: [frames per file]} with files sorted by name, then split: the last file of a label
    # with 2+ files is validation. Split by file, never by frame: neighbouring frames are
    # near-identical, and a frame split would grade the model on its own training data.
    train, val = {}, {}
    for label in labels:
        files = [read_ndjson(f) for f in sorted((DATA / kind / label).glob("*.ndjson"))]
        cut = len(files) - 1 if len(files) >= 2 else len(files)
        train[label], val[label] = files[:cut], files[cut:]
    return train, val


def trail_point(points):
    # Same as palmCentre and handSpan in tui/src/pi: change one, change both.
    centre = points[[0, 5, 9, 13, 17]].mean(0)
    span = max(float(np.linalg.norm(points[9] - points[0])), 1e-6)
    return np.array([centre[0], centre[1], span], np.float32)


def clips(frames):
    # A recording split on frames without a hand: each run is one performance.
    runs, run = [], []
    for _, points in frames + [(None, None)]:
        if points is not None:
            run.append(trail_point(points))
        elif run:
            runs.append(np.stack(run))
            run = []
    return [r for r in runs if len(r) >= 4]


def pose_set(per_label):
    xs = [p for files in per_label.values() for frames in files for _, p in frames if p is not None]
    ys = [GESTURES.index(l) for l, files in per_label.items() for frames in files for _, p in frames if p is not None]
    return (np.stack(xs) if xs else np.zeros((0, 21, 2), np.float32)), np.array(ys, np.int64)


def clip_set(per_label):
    return [(c, MOTIONS.index(l)) for l, files in per_label.items() for frames in files for c in clips(frames)]


static_train, static_val = takes("static", GESTURES)
motion_train, motion_val = takes("motion", MOTIONS)
own_X, own_y = pose_set(static_train)
val_X, val_y = pose_set(static_val)
motion_clips, motion_val_clips = clip_set(motion_train), clip_set(motion_val)
# A held pose is not a motion: every static take is free `none` for the motion head.
motion_clips += [(c, NONE_MOTION) for files in static_train.values() for frames in files for c in clips(frames)]

print(f"{'':12} {'train':>7} {'val':>6}   frames (static) / clips (motion)")
for label in GESTURES:
    print(f"{label:12} {(own_y == GESTURES.index(label)).sum():7} {(val_y == GESTURES.index(label)).sum():6}")
for label in MOTIONS:
    i = MOTIONS.index(label)
    print(f"{label:12} {sum(y == i for _, y in motion_clips):7} {sum(y == i for _, y in motion_val_clips):6}")

for label, axis, sign in [("swipe_left", 0, -1), ("swipe_right", 0, 1), ("swipe_up", 1, -1), ("swipe_down", 1, 1)]:
    moves = [c[-1, axis] - c[0, axis] for c, y in motion_clips if y == MOTIONS.index(label)]
    if moves and np.median(moves) * sign < 0:
        print(f"⚠ {label}: clips mostly move the other way in the image — mirrored camera? see the note above")

               train    val   frames (static) / clips (motion)
none               0      0
pinch              0      0
palm               0      0
fist               0      0
point              0      0
peace              0      0
thumbs_up          0      0
thumbs_down        0      0
ok                 0      0
three              0      0
call               0      0
rock               0      0
none               0      0
swipe_left         0      0
swipe_right        0      0
swipe_up           0      0
swipe_down         0      0
wave               0      0


## 2. HaGRID (optional pretraining)

HaGRID v2's annotation archive carries MediaPipe landmarks for ~1M hands, so no images are
needed: a one-off 720 MB download, parsed once into `data/hagrid/hagrid.npz`. Its coordinates
are normalised to an image size it does not record; the loader assumes landscape 16:9 and the
aspect augmentation below absorbs the portrait shots that gets wrong. It is selfie-style, not a
top-down bench camera, so your own validation takes are the judge: if accuracy there is better
with `USE_HAGRID = False`, keep it off.

Licence: a CC BY-SA 4.0 variant; read its terms before shipping a model trained on it.

In [3]:
def load_hagrid():
    cache = DATA / "hagrid" / "hagrid.npz"
    if cache.exists():
        saved = np.load(cache)
        return saved["X"], saved["y"]

    archive = DATA / "hagrid" / "annotations.zip"
    if not archive.exists():
        archive.parent.mkdir(parents=True, exist_ok=True)
        print("downloading HaGRID annotations (~720 MB, landmarks only)…")
        partial = archive.with_suffix(".part")
        with urllib.request.urlopen(HAGRID_URL) as response, open(partial, "wb") as out:
            shutil.copyfileobj(response, out)
        partial.rename(archive)  # never leave a truncated zip that looks complete

    by_class = {}
    with zipfile.ZipFile(archive) as z:
        # One file per HaGRID class, so unmapped classes are skipped without being parsed.
        names = [n for n in z.namelist() if n.endswith(".json") and ".ipynb_checkpoints" not in n]
        for name in names:
            if Path(name).stem not in HAGRID_MAP:
                continue
            for ann in json.loads(z.read(name)).values():
                for label, marks in zip(ann.get("labels", []), ann.get("hand_landmarks", [])):
                    if label in HAGRID_MAP and len(marks) == 21:
                        by_class.setdefault(HAGRID_MAP[label], []).append(np.asarray(marks, np.float32))

    xs, ys = [], []
    for label, marks in by_class.items():
        pick = rng.choice(len(marks), min(HAGRID_PER_CLASS, len(marks)), replace=False)
        xs.append(np.stack([marks[i] for i in pick]))
        ys += [GESTURES.index(label)] * len(pick)
    X, y = np.concatenate(xs), np.array(ys, np.int64)
    X[..., 0] *= 16 / 9
    np.savez_compressed(cache, X=X, y=y)
    return X, y


hagrid_X, hagrid_y = load_hagrid() if USE_HAGRID else (np.zeros((0, 21, 2), np.float32), np.zeros(0, np.int64))
print("HaGRID:", {GESTURES[i]: int(n) for i, n in enumerate(np.bincount(hagrid_y, minlength=len(GESTURES))) if n})

downloading HaGRID annotations (~720 MB, landmarks only)…
HaGRID: {'none': 3000, 'palm': 3000, 'fist': 3000, 'point': 3000, 'peace': 3000, 'thumbs_up': 3000, 'thumbs_down': 3000, 'ok': 3000, 'three': 3000, 'call': 3000, 'rock': 3000}


## 3. Model

Everything a consumer would otherwise have to reimplement is in `forward`:

- **Pose**: wrist-relative and divided by wrist→middle-knuckle length (`handSpan` in
  `gestures.ts`), so position and distance from the camera do not matter. MLP 42→64→64→12.
- **Motion**: positions relative to the newest frame over mean span, plus log span change
  (towards / away from the camera). Two small 1-D convolutions, mean-pooled.

In [4]:
class HandGesture(nn.Module):
    def __init__(self):
        super().__init__()
        self.pose = nn.Sequential(
            nn.Linear(42, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, len(GESTURES)),
        )
        self.move = nn.Sequential(
            nn.Conv1d(3, 32, 3, padding=1), nn.ReLU(),
            nn.Conv1d(32, 32, 3, padding=1), nn.ReLU(),
        )
        self.move_out = nn.Linear(32, len(MOTIONS))

    def pose_logits(self, landmarks):  # [N, 21, 2] pixels
        points = landmarks - landmarks[:, :1]
        span = points[:, 9].norm(dim=-1).clamp_min(1e-6)
        return self.pose((points / span[:, None, None]).flatten(1))

    def motion_logits(self, trail):  # [N, T, 3] (x, y, span) pixels
        span = trail[:, :, 2:].clamp_min(1e-6)
        moved = (trail[:, :, :2] - trail[:, -1:, :2]) / span.mean(dim=1, keepdim=True)
        x = torch.cat([moved, torch.log(span / span[:, -1:])], dim=-1).transpose(1, 2)
        return self.move_out(self.move(x).mean(dim=-1))

    def forward(self, landmarks, trail):
        return self.pose_logits(landmarks).softmax(-1), self.motion_logits(trail).softmax(-1)


model = HandGesture()
print(sum(p.numel() for p in model.parameters()), "weights")

11314 weights


## 4. Augmentation

The graph already removes position and scale, so augmentation only covers what it cannot:
mirroring (left and right hands, mirrored feeds), roll (±25°, small enough that thumbs up and
down stay apart), aspect (±40% stretch: HaGRID's unknown aspect and a top-down camera's
perspective) and landmark jitter. Motion also gets speed (0.5–2×: other frame rates, and the
remote client, whose echoed trail skips frames).

In [5]:
SWAP = {MOTIONS.index(a): MOTIONS.index(b) for a, b in [("swipe_left", "swipe_right"), ("swipe_right", "swipe_left")]
        if a in MOTIONS and b in MOTIONS}


def rotate(xy, angle):  # xy [N, K, 2], angle [N]
    c, s = angle.cos()[:, None], angle.sin()[:, None]
    return torch.stack([c * xy[..., 0] - s * xy[..., 1], s * xy[..., 0] + c * xy[..., 1]], -1)


def augment_pose(X):
    n = len(X)
    X = X - X[:, :1]
    X = X * torch.stack([torch.where(torch.rand(n) < 0.5, -1.0, 1.0), torch.ones(n)], -1)[:, None]
    X = rotate(X, (torch.rand(n) * 2 - 1) * math.radians(25))
    X = X * torch.stack([torch.exp((torch.rand(n) * 2 - 1) * math.log(1.4)), torch.ones(n)], -1)[:, None]
    span = X[:, 9].norm(dim=-1).clamp_min(1e-6)
    return X + torch.randn_like(X) * 0.02 * span[:, None, None]


def window_at(clip, end, speed=1.0):
    # WINDOW frames ending at `end` (fractional, interpolated), `speed` clip frames apart.
    # Indices before the clip start clamp to its first frame: the server's left-padding.
    at = np.clip(end - speed * np.arange(WINDOW - 1, -1, -1), 0, len(clip) - 1)
    lo = np.floor(at).astype(int)
    hi = np.minimum(lo + 1, len(clip) - 1)
    w = (at - lo)[:, None]
    return (clip[lo] * (1 - w) + clip[hi] * w).astype(np.float32)


def motion_batch(clips, n):
    by_label = {}
    for clip, label in clips:
        by_label.setdefault(label, []).append(clip)
    labels = list(by_label)
    xs, ys = [], []
    for label in rng.choice(labels, n):  # class-balanced
        group = by_label[label]
        clip = group[rng.integers(len(group))]
        # A gesture clip is labelled by its back half, where the motion has happened.
        end = rng.uniform(0 if label == NONE_MOTION else (len(clip) - 1) / 2, len(clip) - 1)
        window = window_at(clip, end, math.exp(rng.uniform(math.log(0.5), math.log(2))))
        if rng.random() < 0.5:
            window[:, 0] *= -1
            label = SWAP.get(label, label)
        xs.append(window)
        ys.append(label)
    X = torch.tensor(np.stack(xs))
    xy = rotate(X[:, :, :2] - X[:, -1:, :2], (torch.rand(n) * 2 - 1) * math.radians(15))
    xy = xy + torch.randn_like(xy) * 0.03 * X[:, :, 2:].mean(1, keepdim=True)
    return torch.cat([xy, X[:, :, 2:]], -1), torch.tensor(ys)

## 5. Train

Pose: pretrain on HaGRID plus your takes, then fine-tune on your takes at a lower rate. HaGRID
rows for classes you did not record stay in the fine-tune so it does not forget them. Motion:
your takes only. Both are seconds on a laptop CPU.

In [6]:
def class_weights(y, n):
    counts = np.bincount(y, minlength=n).astype(np.float32)
    present = counts > 0
    return torch.tensor(np.where(present, counts.sum() / np.maximum(counts, 1) / present.sum(), 0), dtype=torch.float32)


def train_pose(X, y, epochs, lr):
    X, y = torch.tensor(X), torch.tensor(y)
    opt = torch.optim.AdamW(model.pose.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights(y.numpy(), len(GESTURES)), label_smoothing=0.05)
    model.train()
    for epoch in range(epochs):
        total = 0.0
        for idx in torch.randperm(len(X)).split(256):
            loss = loss_fn(model.pose_logits(augment_pose(X[idx])), y[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * len(idx)
        if epoch % 5 == 4 or epoch == epochs - 1:
            print(f"  pose epoch {epoch + 1}: loss {total / len(X):.3f}")
    model.eval()


def train_motion(clips, steps, lr):
    opt = torch.optim.AdamW([*model.move.parameters(), *model.move_out.parameters()], lr=lr, weight_decay=1e-4)
    loss_fn = nn.CrossEntropyLoss(label_smoothing=0.05)
    model.train()
    for step in range(steps):
        X, y = motion_batch(clips, 128)
        loss = loss_fn(model.motion_logits(X), y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        if step % 500 == 499 or step == steps - 1:
            print(f"  motion step {step + 1}: loss {loss.item():.3f}")
    model.eval()


started = time.time()
if len(hagrid_X) + len(own_X):
    print("pose: pretrain")
    train_pose(np.concatenate([hagrid_X, own_X]), np.concatenate([hagrid_y, own_y]), epochs=30, lr=3e-3)
if len(own_X):
    print("pose: fine-tune on your takes")
    keep = ~np.isin(hagrid_y, np.unique(own_y))
    train_pose(np.concatenate([hagrid_X[keep], own_X]), np.concatenate([hagrid_y[keep], own_y]), epochs=15, lr=5e-4)
motion_trained = any(y != NONE_MOTION for _, y in motion_clips)
if motion_trained:
    print("motion")
    train_motion(motion_clips, steps=2000, lr=3e-3)
else:
    # Pinned to `none` rather than shipped random: a random head can still be confident on
    # some trail, and a confident wrong swipe is a scroll nobody asked for.
    with torch.no_grad():
        for p in [*model.move.parameters(), *model.move_out.parameters()]:
            p.zero_()
        model.move_out.bias[NONE_MOTION] = 10.0
    print("⚠ no motion takes: the motion head always answers none (record data/motion/ to train it)")
print(f"trained in {time.time() - started:.0f}s")

pose: pretrain
  pose epoch 5: loss 0.406
  pose epoch 10: loss 0.366
  pose epoch 15: loss 0.350
  pose epoch 20: loss 0.338
  pose epoch 25: loss 0.334
  pose epoch 30: loss 0.331
⚠ no motion takes: the motion head is untrained
trained in 7s


## 6. Evaluate on the held-out takes

In [7]:
def report(title, labels, truth, guess):
    if len(truth) == 0:
        print(f"{title}: no held-out takes (a label needs 2+ files to hold one out)\n")
        return
    n = len(labels)
    matrix = np.zeros((n, n), int)
    np.add.at(matrix, (truth, guess), 1)
    print(f"{title}: {np.mean(truth == guess):.1%} of {len(truth)}   (rows: truth, columns: predicted)")
    width = max(map(len, labels)) + 3
    print(" " * width, *(str(i).rjust(5) for i in range(n)))
    for i in range(n):
        if matrix[i].sum():
            row = f"{i} {labels[i]}".rjust(width)
            print(row, *(str(v).rjust(5) for v in matrix[i]), f"  {matrix[i, i] / matrix[i].sum():.0%}")
    print()


with torch.no_grad():
    report("pose", GESTURES, val_y, model.pose_logits(torch.tensor(val_X)).argmax(-1).numpy() if len(val_X) else val_y)
    if motion_val_clips:
        windows = torch.tensor(np.stack([window_at(c, len(c) - 1) for c, _ in motion_val_clips]))
        truth = np.array([y for _, y in motion_val_clips])
        report("motion", MOTIONS, truth, model.motion_logits(windows).argmax(-1).numpy())
    else:
        report("motion", MOTIONS, np.zeros(0, int), np.zeros(0, int))

pose: no held-out takes (a label needs 2+ files to hold one out)

motion: no held-out takes (a label needs 2+ files to hold one out)



## 7. Export

Fixed batch-1 shapes (one hand per frame, which is what the tracker returns) and opset 18, so
older runtimes, `onnxruntime-web` and mobile load it too. Then: ONNX Runtime must agree with
PyTorch, and the latency is measured single-threaded, the way `track.ts` runs it.

In [8]:
OUT.mkdir(exist_ok=True)
onnx_path = OUT / "hand_gesture.onnx"
sample_landmarks = np.concatenate([val_X, own_X, hagrid_X])[:64] if len(val_X) + len(own_X) + len(hagrid_X) else rng.uniform(0, 640, (64, 21, 2)).astype(np.float32)
sample_trails = (motion_batch(motion_clips, 64)[0].numpy() if motion_clips else rng.uniform(1, 640, (64, WINDOW, 3)).astype(np.float32))

torch.onnx.export(
    model.eval(),
    (torch.tensor(sample_landmarks[:1]), torch.tensor(sample_trails[:1])),
    onnx_path,
    input_names=["landmarks", "trail"],
    output_names=["gesture", "motion"],
    opset_version=18,
    external_data=False,
)
(OUT / "hand_gesture.json").write_text(json.dumps({"gesture": GESTURES, "motion": MOTIONS, "window": WINDOW}))

options = ort.SessionOptions()
options.intra_op_num_threads = 1
options.inter_op_num_threads = 1
session = ort.InferenceSession(onnx_path, options, providers=["CPUExecutionProvider"])

worst = 0.0
with torch.no_grad():
    for landmarks, trail in zip(sample_landmarks, sample_trails):
        feeds = {"landmarks": landmarks[None], "trail": trail[None].astype(np.float32)}
        ours = session.run(None, feeds)
        theirs = model(torch.tensor(feeds["landmarks"]), torch.tensor(feeds["trail"]))
        worst = max(worst, *(float(np.abs(a - b.numpy()).max()) for a, b in zip(ours, theirs)))
assert worst < 1e-4, f"ONNX Runtime disagrees with PyTorch by {worst}"

feeds = {"landmarks": sample_landmarks[:1], "trail": sample_trails[:1].astype(np.float32)}
for _ in range(200):
    session.run(None, feeds)
timings = []
for _ in range(2000):
    t0 = time.perf_counter()
    session.run(None, feeds)
    timings.append(time.perf_counter() - t0)
print(f"parity: max diff {worst:.1e}")
print(f"size:   {onnx_path.stat().st_size / 1024:.0f} KB")
print(f"speed:  {np.median(timings) * 1e6:.0f} µs median, {np.percentile(timings, 99) * 1e6:.0f} µs p99 (1 thread)")

W1009 08:44:14.947000 11165 torch/onnx/_internal/exporter/_registration.py:107] torchvision is not installed. Skipping torchvision::nms
W1009 08:44:14.948000 11165 torch/onnx/_internal/exporter/_registration.py:107] torchvision is not installed. Skipping torchvision::roi_align
W1009 08:44:14.948000 11165 torch/onnx/_internal/exporter/_registration.py:107] torchvision is not installed. Skipping torchvision::roi_pool
W1009 08:44:14.949000 11165 torch/onnx/_internal/exporter/_registration.py:107] torchvision is not installed. Skipping torchvision::deform_conv2d


[torch.onnx] Obtain model graph for `HandGesture([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `HandGesture([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
parity: max diff 2.4e-07
size:   77 KB
speed:  16 µs median, 27 µs p99 (1 thread)


## 8. Install

Copies the pair to the web API (`web/models/hand/`, commit them: they are ours and tiny) and to
the Pi worker's model directory. Both pick them up on their next start.

HaGRID alone is enough to install: it is selfie-style, like a laptop webcam, and covers every pose
but pinch. Without motion takes the motion head is pinned to `none`, so nothing swipes by accident.

In [11]:
assert len(own_X) or len(hagrid_X), "nothing trained: record takes (see 1. Record) or set USE_HAGRID"
if not len(own_X):
    print("⚠ HaGRID only: fine for a laptop webcam, no pinch, and check it on the real camera")
if not motion_trained:
    print("⚠ no motion takes: swipes and wave will never fire")
data_home = Path(os.environ.get("XDG_DATA_HOME") or Path.home() / ".local" / "share")
for target in [HERE.parent / "web" / "models" / "hand", data_home / "jarvis" / "models"]:
    target.mkdir(parents=True, exist_ok=True)
    for name in ("hand_gesture.onnx", "hand_gesture.json"):
        shutil.copy(OUT / name, target / name)
    print("→", target)

⚠ HaGRID only: fine for a laptop webcam, no pinch, and check it on the real camera
⚠ no motion takes: swipes and wave will never fire
→ /Users/mralev/Code/JARVIS/web/models/hand
→ /Users/mralev/.local/share/jarvis/models
